## 1.Import Libraries

In [133]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## 2.Import Dataset

In [110]:
data = pd.read_csv('loan_dataset.csv')

In [65]:
data.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [66]:
data.shape

(614, 13)

In [67]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    object 
 1   Gender             601 non-null    object 
 2   Married            611 non-null    object 
 3   Dependents         599 non-null    object 
 4   Education          614 non-null    object 
 5   Self_Employed      582 non-null    object 
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    object 
 12  Loan_Status        614 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 62.5+ KB


In [68]:
data.describe()

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
count,614.000000,614.000000,592.000000,600.00000,564.000000
mean,5403.459283,1621.245798,146.412162,342.00000,0.842199
std,6109.041673,2926.248369,85.587325,65.12041,0.364878
min,150.000000,0.000000,9.000000,12.00000,0.000000
25%,2877.500000,0.000000,100.000000,360.00000,1.000000
50%,3812.500000,1188.500000,128.000000,360.00000,1.000000
75%,5795.000000,2297.250000,168.000000,360.00000,1.000000
max,81000.000000,41667.000000,700.000000,480.00000,1.000000


In [69]:
categorical_variables = ['Gender','Married','Dependents','Education','Self_Employed','Credit_History', 'Property_Area', 'Loan_Status']
print('Cateforical variable are :\n')
for i in range(len(categorical_variables)):
    print(categorical_variables[i])

Cateforical variable are :

Gender
Married
Dependents
Education
Self_Employed
Credit_History
Property_Area
Loan_Status


## 3.Replace the missing values

In [70]:
data.isnull().sum()

Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64

In [111]:
# 用眾數填充類別型資料的缺失值

data.Gender.fillna(data.Gender.mode()[0], inplace=True)
data.Married.fillna(data.Married.mode()[0], inplace=True)
data.Dependents.fillna(data.Dependents.mode()[0], inplace=True)
data.Self_Employed.fillna(data.Self_Employed.mode()[0], inplace=True)
data.Credit_History.fillna(data.Credit_History.mode()[0], inplace=True)

In [112]:
# 用中位數填充數值型資料的缺失值

data.LoanAmount.fillna(data.LoanAmount.median(), inplace=True)
data.Loan_Amount_Term.fillna(data.Loan_Amount_Term.median(), inplace=True)

In [113]:
data.isnull().sum()

Loan_ID              0
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64

## 4.One-Hot Encoding

In [115]:
# 男性為1，女性為0
data['Gender'] = data['Gender'].replace(['Male'], 1)
data['Gender'] = data['Gender'].replace(['Female'], 0)
# 已婚為1，未婚為0
data['Married'] = data['Married'].replace(['Yes'], 1)
data['Married'] = data['Married'].replace(['No'], 0)

# 家屬人數
dummies_Dependents = pd.get_dummies(data['Dependents'])
dummies_Dependents = dummies_Dependents.rename(columns = {'0': 'Dependents_0','1': 'Dependents_1','2': 'Dependents_2','3+': 'Dependents_3+'})
dummies_Dependents['Dependents_0'] = dummies_Dependents['Dependents_0'].astype(int)
dummies_Dependents['Dependents_1'] = dummies_Dependents['Dependents_1'].astype(int)
dummies_Dependents['Dependents_2'] = dummies_Dependents['Dependents_2'].astype(int)
dummies_Dependents['Dependents_3+'] = dummies_Dependents['Dependents_3+'].astype(int)

# 教育程度
dummies_Education = pd.get_dummies(data['Education'])
# 畢業為1，未畢業為0
dummies_Education['Graduate'] = dummies_Education['Graduate'].astype(int)

# 自雇為1，受雇為0
data['Self_Employed'] = data['Self_Employed'].replace(['Yes'],1)
data['Self_Employed'] = data['Self_Employed'].replace(['No'],0)

# 房產區域
dummies_PropertyArea = pd.get_dummies(data['Property_Area'])
dummies_PropertyArea['Rural'] = dummies_PropertyArea['Rural'].astype(int)
dummies_PropertyArea['Semiurban'] = dummies_PropertyArea['Semiurban'].astype(int)
dummies_PropertyArea['Urban'] = dummies_PropertyArea['Urban'].astype(int)


In [116]:
data= pd.concat([data,dummies_Dependents],axis=1)
data= pd.concat([data,dummies_Education],axis=1)
data= pd.concat([data,dummies_PropertyArea],axis=1)

In [117]:
# data=data.drop(['Loan_ID','Gender','Married','Property_Area','Dependents','Self_Employed','Education','NotMarried','NotSelf_Employed','Not Graduate','Female'] , axis=1)
data = data.drop(['Loan_ID','Property_Area','Dependents','Education','Not Graduate'] , axis=1)


data.head()

,Gender,Married,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Loan_Status,Dependents_0,Dependents_1,Dependents_2,Dependents_3+,Graduate,Rural,Semiurban,Urban
0,1,0,0,5849,0.0,128.0,360.0,1.0,Y,1,0,0,0,1,0,0,1
1,1,1,0,4583,1508.0,128.0,360.0,1.0,N,0,1,0,0,1,1,0,0
2,1,1,1,3000,0.0,66.0,360.0,1.0,Y,1,0,0,0,1,0,0,1
3,1,1,0,2583,2358.0,120.0,360.0,1.0,Y,1,0,0,0,0,0,0,1
4,1,0,0,6000,0.0,141.0,360.0,1.0,Y,1,0,0,0,1,0,0,1


In [ ]:
# 調整欄位順序
data_shift = data.loc[:, ['Gender', 'Married', 'Self_Employed']]
data = data.drop(columns=['Gender', 'Married', 'Self_Employed'], axis=1)
data.insert(6, 'Gender', data_shift.loc[:, ['Gender']])
data.insert(7, 'Married', data_shift.loc[:, ['Married']])
data.insert(13, 'Self_Employedied', data_shift.loc[:, ['Self_Employed']])
data

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Loan_Status,Gender,Married,Dependents_0,Dependents_1,Dependents_2,Dependents_3+,Graduate,Self_Employedied,Rural,Semiurban,Urban
0,5849,0.0,128.0,360.0,1.0,Y,1,0,1,0,0,0,1,0,0,0,1
1,4583,1508.0,128.0,360.0,1.0,N,1,1,0,1,0,0,1,0,1,0,0
2,3000,0.0,66.0,360.0,1.0,Y,1,1,1,0,0,0,1,1,0,0,1
3,2583,2358.0,120.0,360.0,1.0,Y,1,1,1,0,0,0,0,0,0,0,1
4,6000,0.0,141.0,360.0,1.0,Y,1,0,1,0,0,0,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
609,2900,0.0,71.0,360.0,1.0,Y,0,0,1,0,0,0,1,0,1,0,0
610,4106,0.0,40.0,180.0,1.0,Y,1,1,0,0,0,1,1,0,1,0,0
611,8072,240.0,253.0,360.0,1.0,Y,1,1,0,1,0,0,1,0,0,0,1
612,7583,0.0,187.0,360.0,1.0,Y,1,1,0,0,1,0,1,0,0,0,1


In [121]:
# 把貸款狀態Y/N改成1/0

data['Loan_Status']= data['Loan_Status'].replace(['Y'],1)
data['Loan_Status']= data['Loan_Status'].replace(['N'],0)

In [122]:
data

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Loan_Status,Gender,Married,Dependents_0,Dependents_1,Dependents_2,Dependents_3+,Graduate,Self_Employedied,Rural,Semiurban,Urban
0,5849,0.0,128.0,360.0,1.0,1,1,0,1,0,0,0,1,0,0,0,1
1,4583,1508.0,128.0,360.0,1.0,0,1,1,0,1,0,0,1,0,1,0,0
2,3000,0.0,66.0,360.0,1.0,1,1,1,1,0,0,0,1,1,0,0,1
3,2583,2358.0,120.0,360.0,1.0,1,1,1,1,0,0,0,0,0,0,0,1
4,6000,0.0,141.0,360.0,1.0,1,1,0,1,0,0,0,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
609,2900,0.0,71.0,360.0,1.0,1,0,0,1,0,0,0,1,0,1,0,0
610,4106,0.0,40.0,180.0,1.0,1,1,1,0,0,0,1,1,0,1,0,0
611,8072,240.0,253.0,360.0,1.0,1,1,1,0,1,0,0,1,0,0,0,1
612,7583,0.0,187.0,360.0,1.0,1,1,1,0,0,1,0,1,0,0,0,1


## 5.Splitting the data set to Training data and Test data 

In [123]:
# 區分特徵與標籤資料，'Loan_Status'欄位為是否通過貸款的標籤資料

X=data.drop(columns='Loan_Status')
Y=pd.DataFrame(data['Loan_Status'])

In [124]:
# 切割85%的訓練資料與15%的測試資料
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.15,random_state=3)

In [125]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   ApplicantIncome    614 non-null    int64  
 1   CoapplicantIncome  614 non-null    float64
 2   LoanAmount         614 non-null    float64
 3   Loan_Amount_Term   614 non-null    float64
 4   Credit_History     614 non-null    float64
 5   Loan_Status        614 non-null    int64  
 6   Gender             614 non-null    int64  
 7   Married            614 non-null    int64  
 8   Dependents_0       614 non-null    int32  
 9   Dependents_1       614 non-null    int32  
 10  Dependents_2       614 non-null    int32  
 11  Dependents_3+      614 non-null    int32  
 12  Graduate           614 non-null    int32  
 13  Self_Employedied   614 non-null    int64  
 14  Rural              614 non-null    int32  
 15  Semiurban          614 non-null    int32  
 16  Urban              614 non

In [126]:
X_train.info() # 資料筆數為data的85%，且沒有Loan_Status欄位

<class 'pandas.core.frame.DataFrame'>
Index: 521 entries, 276 to 249
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   ApplicantIncome    521 non-null    int64  
 1   CoapplicantIncome  521 non-null    float64
 2   LoanAmount         521 non-null    float64
 3   Loan_Amount_Term   521 non-null    float64
 4   Credit_History     521 non-null    float64
 5   Gender             521 non-null    int64  
 6   Married            521 non-null    int64  
 7   Dependents_0       521 non-null    int32  
 8   Dependents_1       521 non-null    int32  
 9   Dependents_2       521 non-null    int32  
 10  Dependents_3+      521 non-null    int32  
 11  Graduate           521 non-null    int32  
 12  Self_Employedied   521 non-null    int64  
 13  Rural              521 non-null    int32  
 14  Semiurban          521 non-null    int32  
 15  Urban              521 non-null    int32  
dtypes: float64(4), int32(8), int6

## 6.Random Forest Classifier

In [127]:
classifier = RandomForestClassifier(n_estimators=1000,max_features=15,max_depth=5,bootstrap=True)
classifier.fit(X_train,Y_train)
predictions = classifier.predict(X_test)
accuracyScores = accuracy_score(predictions, Y_test)
print(accuracyScores)

c:\Users\Administrator\anaconda3\envs\tfgpu\lib\site-packages\sklearn\base.py:1151: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


0.8709677419354839


In [128]:
df = pd.DataFrame(predictions)
len(df)

93

In [129]:
len(X_test)

93

## 7.Save the Model

In [ ]:
import pickle

filename = 'Random_Forest.sav'
pickle.dump(classifier, open(filename, 'wb'))